In [1]:
library(googledrive)
drive_auth()

Is it OK to cache OAuth access credentials in the folder ~/.cache/gargle
between R sessions?
1: Yes
2: No


Selection: 1


Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=f887513ef0c1332b5aac96c4f80a3f7a&access_type=offline&prompt=consent



Enter authorization code: eyJjb2RlIjoiNC8wQWVvV3VNOWwzUWY2UDdmQ21KUklxUGJHSVAtM1NEMWZIRjJxV3lHOE02SjdZZnlhc3pTejBvRm9GR2FpdzVKZ3B5UnI1dyIsInN0YXRlIjoiZjg4NzUxM2VmMGMxMzMyYjVhYWM5NmM0ZjgwYTNmN2EifQ==


In [2]:
system("sudo apt-get install libgmp-dev")
system("sudo apt-get install libmagick++-dev")
system("sudo apt-get install libgsl-dev")
system("sudo apt-get install libmpfr-dev")
system("sudo apt-get install libgtk-3-dev libcairo2-dev")

.required_pkgs <- c(
  "EnvStats", "psych", "expm",
  "Rfast", "foreach", "doParallel", "devtools"
)
for (.pkg in .required_pkgs) {
  if (!requireNamespace(.pkg, quietly = TRUE)) {
    if (.pkg == "Rfast") {
      install.packages(.pkg, INSTALL_opts = "--no-lock")
    } else {
      install.packages(.pkg)
    }
  }
}

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘nortest’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘mnormt’, ‘GPArotation’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘zigg’, ‘RcppParallel’, ‘RcppArmadillo’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘iterators’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [3]:
library(devtools)
install.packages("remotes")

Loading required package: usethis

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [4]:
##install.packages("/content/rdetools_1.0.tar.gz", repos = NULL, type="source")
# Instala rrcov desde la version local modificada (1.7-7.9000).
# Solo reinstala si la version instalada es distinta a la local.

  devtools::install_local("/content/rrcov.zip", upgrade = "never", quiet = TRUE)



Warning message:
“`install_local()` was deprecated in devtools 2.5.0.
ℹ Please use pak::pak("local::path") instead.”
Installing 4 packages: mvtnorm, DEoptimR, pcaPP, robustbase



In [5]:
library(MASS)
library(rrcov)
library(EnvStats)
library(psych)
library(expm)
library(Rfast)
library(foreach)
library(doParallel)

Loading required package: robustbase

Scalable Robust Estimators with High Breakdown Point (version 1.7-7)



Attaching package: ‘EnvStats’


The following object is masked from ‘package:rrcov’:

    predict


The following object is masked from ‘package:MASS’:

    boxcox


The following objects are masked from ‘package:stats’:

    predict, predict.lm


The following object is masked from ‘package:base’:

    print.default


Loading required package: Matrix


Attaching package: ‘expm’


The following object is masked from ‘package:Matrix’:

    expm


The following object is masked from ‘package:rrcov’:

    sqrtm


Loading required package: Rcpp

Loading required package: zigg

Loading required package: RcppParallel


Attaching package: ‘RcppParallel’


The following object is masked from ‘package:Rcpp’:

    LdFlags



Rfast: 2.1.5.2

 ___ __ __ __ __    __ __ __ __ __ _             _               __ __ __ __ __     __ __ __ __ __ __   
|  __ __ __ __  |  |  __ __ __ __ _/        

In [6]:
MethodUCLKernel <- function(T2, alpha)
{
  EstKernelSmooth = density(T2,kernel="gaussian",bw="nrd")
  valuePairs = cbind(EstKernelSmooth$x,EstKernelSmooth$y)
  valuePairs = valuePairs[valuePairs[,1]>=0,]
  Estand = valuePairs[,2]/sum(valuePairs[,2])
  NumberRow = nrow(valuePairs)
  countValues = 0

  for(i in 1:NumberRow)
  {
    if(countValues <  (1-alpha))
    {
      j = i
      countValues = sum(Estand[1:i])
    }
  }

  UCL = valuePairs[j,1]
  return(list(UCL=UCL,alpha1=(1-countValues)))
}

SignalProbability <- function (matrixT2, ucl, uclMax, uclKernel )
{

  SignalCountUCL = 0
  SignalCountMax = 0
  SignalCountKernel = 0

  for (i in 1: dim(matrixT2)[1])
  {
    RowMatrix = matrixT2[i,]

    ListFilterUCL = Filter(function(x) x > ucl,RowMatrix)
    SignalCountUCL = SignalCountUCL + length(ListFilterUCL)


    ListFilterMax = Filter(function(x) x > uclMax,RowMatrix)
    SignalCountMax = SignalCountMax + length(ListFilterMax)

    ListFilterKernel = Filter(function(x) x > uclKernel,RowMatrix)
    SignalCountKernel = SignalCountKernel + length(ListFilterKernel)
  }

  TotalValue = dim(matrixT2)[1]*dim(matrixT2)[2]

  SignalPro = SignalCountUCL/TotalValue
  SignalProMax = SignalCountMax/TotalValue
  SignalProKernel = SignalCountKernel/TotalValue

  return(list("SignalPro" = SignalPro,
              "SignalProMax" = SignalProMax,
              "SignalProKernel" = SignalProKernel))
}

AlgorithmMDPCFPart1 <- function(DatesNorm, NumVariable, Observations, Alpha = 0.05, outliersData = c(), OutliersFlag = FALSE) {
  MiddlePoint <- floor(Observations / 2) + 1
  AlgorithmObjects <- AlgorithmRoMDP(DatesNorm)
  # calculo de Media y matrices de correlacion y de varianza
  MeanDMP <- unlist(AlgorithmObjects[1])
  SigmaDMP <- matrix(diag(as.numeric(unlist(AlgorithmObjects[2]))), ncol = NumVariable)
  InverseSigmaDMP <- solve(SigmaDMP)
  SigmaOfMinimunDet <- AlgorithmObjects[3][[1]]
  CorMatrix <- sqrt(InverseSigmaDMP) %*% SigmaOfMinimunDet %*% sqrt(InverseSigmaDMP)

  # Estimacion objetos algoritmo Ebadi
  TraceRhoSquare <- tr(CorMatrix %^% 2) - (NumVariable**2) / MiddlePoint
  TraceRhoCubic <- tr(CorMatrix %^% 3) - ((3 * NumVariable) / MiddlePoint) * tr(CorMatrix %^% 2) + ((2 * (NumVariable**3)) / (MiddlePoint**2))

  # Estimadores Ui y Zi
  ListEstimUi <- c()
  ListEstimZi <- c()
  ListOutliers <- c()
  Zalpha <- qnorm(1 - Alpha, 0, 1)
  ConstantMDP <- 1 + (2 * NumVariable) / (Observations * sqrt(TraceRhoSquare))


  if (OutliersFlag) {
    for (i in 1:dim(outliersData)[1])
    {
      DistanceMahalanobisXi <- t((outliersData[i, ] - MeanDMP)) %*% InverseSigmaDMP %*% (outliersData[i, ] - MeanDMP)
      Ui <- (DistanceMahalanobisXi - NumVariable) / (2 * ConstantMDP * sqrt(TraceRhoSquare))
      ListEstimUi <- c(ListEstimUi, Ui)
      Zi <- Ui - (4 * TraceRhoCubic * (Zalpha**2 - 1)) / (3 * (2 * TraceRhoSquare)^(3 / 2))
      ListEstimZi <- c(ListEstimZi, Zi)
    }
  } else {
    for (i in 1:dim(DatesNorm)[1])
    {
      DistanceMahalanobisXi <- t((DatesNorm[i, ] - MeanDMP)) %*% InverseSigmaDMP %*% (DatesNorm[i, ] - MeanDMP)
      Ui <- (DistanceMahalanobisXi - NumVariable) / (2 * ConstantMDP * sqrt(TraceRhoSquare))
      ListEstimUi <- c(ListEstimUi, Ui)
      Zi <- Ui - (4 * TraceRhoCubic * (Zalpha**2 - 1)) / (3 * (2 * TraceRhoSquare)^(3 / 2))
      ListEstimZi <- c(ListEstimZi, Zi)
    }
  }
  return(list("Ui" = ListEstimUi, "Zi" = ListEstimZi))
}

AlgorithmRoMDP <- function(DataSet, itertime = 100) {
    SampleNumber <- dim(DataSet)[1]
    VariableNumber <- dim(DataSet)[2]
    MiddlePoint <- round(SampleNumber / 2) + 1
    TransposedDataSet <- t(DataSet)

    ValueInitialSubsets <- 2
    BestDet <- 0
    VecZero <- numeric(SampleNumber)

    # Ciclo de iteracciones por default 100
    for (iter in 1:itertime) {
        Id <- sample(SampleNumber, ValueInitialSubsets, replace = FALSE)
        SubsetById <- DataSet[Id, ]
        MuSubsetById <- Rfast::colmeans(SubsetById)
        VarSubsetById <- Rfast::colVars(SubsetById)
        Sama <- (TransposedDataSet - MuSubsetById) / VarSubsetById
        Distance <- Rfast::colsums(Sama)
        Criterio <- 10
        Count <- 0
        while (Criterio != 0 & Count <= 15) {
            Count <- Count + 1
            VectorZeroi <- numeric(SampleNumber)
            DistancePerm <- order(Distance)
            VectorZeroi[DistancePerm[1:MiddlePoint]] <- 1
            Criterio <- sum(abs(VectorZeroi - VecZero))
            VecZero <- VectorZeroi
            NewDataSet <- DataSet[DistancePerm[1:MiddlePoint], ]
            MuSubsetById <- Rfast::colmeans(NewDataSet)
            VarSubsetById <- Rfast::colVars(NewDataSet)
            Sama <- (TransposedDataSet - MuSubsetById) / VarSubsetById
            Distance <- Rfast::colsums(Sama)
        }
        TempDet <- prod(VarSubsetById)
        if (BestDet == 0 | TempDet < BestDet) {
            BestDet <- TempDet
            FinalVec <- VecZero
        }
    }
    SubMCD <- (1:SampleNumber)[FinalVec != 0]

    MuSubsetById <- Rfast::colmeans(DataSet[SubMCD, ])
    VarSubsetById <- Rfast::colVars(DataSet[SubMCD, ])
    Sigma <- cov(DataSet[SubMCD, ])
    return(list(MuSubsetById, VarSubsetById, Sigma, SubMCD))
}

#' Calcular Mahalanobis con TPU (vectorizado)
mahalanobis_tpu_batch <- function(x_matrix, center, cov_inv, use_tpu = FALSE) {
  if (use_tpu && exists("jnp")) {
    tryCatch({
      x_jax <- jnp$array(x_matrix)
      center_jax <- jnp$array(center)
      cov_inv_jax <- jnp$array(cov_inv)

      diff <- x_jax - center_jax
      result <- jnp$sum(diff * jnp$dot(diff, cov_inv_jax), axis = 1L)

      return(py_to_r(result))
    }, error = function(e) {
      return(mahalanobis(x_matrix, center, cov_inv))
    })
  } else {
    return(mahalanobis(x_matrix, center, cov_inv))
  }
}



#' Generar datos multivariados con TPU
rmvnorm_tpu <- function(n, mean, Sigma, use_tpu = FALSE) {
  if (use_tpu && exists("jnp")) {
    tryCatch({
      # Usar JAX random
      key <- jax$random$PRNGKey(as.integer(runif(1, 0, 1e6)))
      mean_jax <- jnp$array(mean)
      sigma_jax <- jnp$array(Sigma)

      # Generar datos en TPU
      result <- jax$random$multivariate_normal(
        key,
        mean_jax,
        sigma_jax,
        shape = as.integer(n)
      )

      return(py_to_r(result))
    }, error = function(e) {
      return(mvrnorm(n, mean, Sigma))
    })
  } else {
    return(mvrnorm(n, mean, Sigma))
  }
}


SimulationT2Chart <- function(observation, numVariables, numSimulation, meanVector, sigmaMatriz, alphaMRCD = 0.75, typeMethod = "MRCD") {
  T2Matrix <- c()
  T2Total <- c()
  T2Max <- c()

  for (i in 1:numSimulation)
  {
    Data <- rmvnorm_tpu(observation, meanVector, sigmaMatriz, TRUE)
    if (typeMethod == "MRCD") {
      CovMRCD <- CovMrcd(Data, alpha = alphaMRCD)
      MediaMRCD <- CovMRCD$center
      SigmaMRCD <- CovMRCD$cov
      BestSubset <- CovMRCD$best

      DataBestSubset <- Data[BestSubset, ]
      T2 <- mahalanobis_tpu_batch(DataBestSubset, center = MediaMRCD, cov = SigmaMRCD, use_tpu = TRUE)
      LenMatrix <- length(BestSubset)
    } else if (typeMethod == "T2MOD") {
      Media <- colMeans(Data)
      Sigma <- cov(Data)
      SigmaMod <- 1 / (sum(diag(Sigma)) / numVariables)

      T2 <- c()
      for (j in 1:dim(Data)[1])
      {
        Ai <- (observation / (observation - 1)) * (norm((Data[j, ] - Media) / sqrt(numVariables)^2, type = "2") / SigmaMod)
        T2 <- c(Ai, T2)
      }
      LenMatrix <- observation
    } else if (typeMethod == "EBADIUI") {
      T2 <- c()
      Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, FALSE)
      T2 <- c(Firstestimation$Ui, T2)
      LenMatrix <- observation
    } else if (typeMethod == "EBADIZI") {
      T2 <- c()
      Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, FALSE)
      T2 <- c(Firstestimation$Zi, T2)
      LenMatrix <- observation
    }
    T2Total <- c(T2Total, T2)
    T2Max <- c(T2Max, max(T2))
  }

  T2Matrix <- matrix(T2Total, ncol = LenMatrix)
  return(list(
    "T2Matrix" = T2Matrix,
    "T2Total" = T2Total,
    "T2Max" = T2Max
  ))
}




SimulationT2ChartOutliers <- function(observation, numVariables,
                                      numSimulation, meanVector,
                                      sigmaMatriz, shiftMean, percentoutliers = 0, alphaMRCD = 0.75, DeltaNCP = 0.05,
                                      UCL = 1, UCLMax = 1, UCLKernel = 1, typeMethod = "MRCD") {
  T2Matrix <- c()
  T2Total <- c()
  T2Max <- c()

  for (i in 1:numSimulation)
  {
    NumOutliers <- floor(percentoutliers * observation)
    if (identical(meanVector, shiftMean)) {
      Data <- rmvnorm_tpu(observation, meanVector, Sigma = sigmaMatriz, use_tpu = TRUE)
    } else {
      Data <- rmvnorm_tpu(observation - NumOutliers, meanVector, Sigma = sigmaMatriz, use_tpu = TRUE)
      if (NumOutliers >= 1) {
        DataOutlier <- rmvnorm_tpu(NumOutliers, shiftMean, Sigma = sigmaMatriz, use_tpu = TRUE)
        Data <- rbind(Data, DataOutlier)
      }
    }
    if (typeMethod == "MRCD") {
      CovMRCD <- CovMrcd(Data, alpha = alphaMRCD)
      MediaMRCD <- CovMRCD$center
      SigmaMRCD <- CovMRCD$cov
      BestSubset <- CovMRCD$best
      DataBestSubset <- Data[BestSubset, ]
      if (identical(meanVector, shiftMean)) {
        T2 <- mahalanobis_tpu_batch(DataBestSubset, center = MediaMRCD, cov = SigmaMRCD, use_tpu = TRUE)
      } else {
        T2 <- mahalanobis_tpu_batch(DataOutlier, center = MediaMRCD, cov = SigmaMRCD, use_tpu = TRUE)
      }
    } else if (typeMethod == "T2MOD") {
      Media <- colMeans(Data)
      Sigma <- cov(Data)
      SigmaMod <- 1 / (sum(diag(Sigma)) / numVariables)

      T2 <- c()
      if (identical(meanVector, shiftMean)) {
        for (j in 1:dim(Data)[1])
        {
          Ai <- (observation / (observation - 1)) * (norm((Data[j, ] - Media) / sqrt(numVariables)^2, type = "2") / SigmaMod)
          T2 <- c(Ai, T2)
        }
      } else {
        for (j in 1:dim(DataOutlier)[1])
        {
          Ai <- (observation / (observation - 1)) * (norm((DataOutlier[j, ] - Media) / sqrt(numVariables)^2, type = "2") / SigmaMod)
          T2 <- c(Ai, T2)
        }
      }
    } else if (typeMethod == "EBADIUI") {
      T2 <- c()

      if (identical(meanVector, shiftMean)) {
        Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, c(), FALSE)
        T2 <- c(Firstestimation$Ui, T2)
      } else {
        Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, DataOutlier, TRUE)
        T2 <- c(Firstestimation$Ui, T2)
      }
      LenMatrix <- observation
    } else if (typeMethod == "EBADIZI") {
      T2 <- c()
      if (identical(meanVector, shiftMean)) {
        Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, c(), FALSE)
        T2 <- c(Firstestimation$Zi, T2)
      } else {
        Firstestimation <- AlgorithmMDPCFPart1(Data, numVariables, observation, 0.05, DataOutlier, TRUE)
        T2 <- c(Firstestimation$Zi, T2)
      }
      LenMatrix <- observation
    }
    T2Total <- c(T2Total, T2)
    T2Max <- c(T2Max, max(T2))
  }
  T2Matrix <- matrix(T2Total, ncol = 1)
  SignalProbabilityT2OutliersMRCD <- SignalProbability(T2Matrix, UCL, UCLMax, UCLKernel)
  return(c(
    DeltaNCP,
    SignalProbabilityT2OutliersMRCD$SignalPro,
    SignalProbabilityT2OutliersMRCD$SignalProMax,
    SignalProbabilityT2OutliersMRCD$SignalProKernel
  ))
}


In [7]:
n_cores_available <- parallel::detectCores()
n_cores_use <- max(1L, n_cores_available - 1L)
cl <- makeCluster(n_cores_use)
registerDoParallel(cl)

set.seed(123)

# Parameters Simulation
for(NumberVariable in c(200,250)){
  Observation <- 100
  Percentoutliers <- 0.2
  NumSimulation <- 10000
  path = paste("/content/SinalprobabilityNormal",Observation,"x",NumberVariable,"x",Percentoutliers,"x",NumSimulation,".RData",sep = "")
  fun <- function(i, j) (0.5)^(abs(i - j))

  rows <- 1:NumberVariable
  cols <- 1:NumberVariable

  SigmaCorr <- outer(rows, cols, FUN = fun)
  mu <- rep(0, NumberVariable)
  AlphaMRCD <- 0.75
  AlphaPFA <- 0.05
  NumSimulationDelta <- 10000

  # Lotes: cada foreach corre SimPerBatch simulaciones en lugar de todas juntas.
  # Permite usar más núcleos en Parte 1 (4 métodos × NumBatches tareas en paralelo)
  # y recuperarse ante fallos sin perder todo el cómputo.
  NumBatches        <- 10L
  SimPerBatch       <- NumSimulation       %/% NumBatches   # 1 000
  SimPerBatchDelta  <- NumSimulationDelta  %/% NumBatches   # 1 000



  # Parte 1 Calculo de los Limites de Probabilidad MRCD
  # Grid plano: 4 métodos × NumBatches = 40 tareas (usan ~40 núcleos en paralelo).

  .local_fns <- c("SimulationT2Chart", "rmvnorm_tpu", "mahalanobis_tpu_batch",
                   "AlgorithmMDPCFPart1", "AlgorithmRoMDP")
  .pkgs <- c("rrcov", "MASS", "Rfast", "EnvStats", "KernSmooth", "psych", "expm")

  .methods1  <- c("MRCD", "T2MOD", "EBADIUI", "EBADIZI")
  .p1_grid   <- expand.grid(method = .methods1, batch = seq_len(NumBatches),
                             stringsAsFactors = FALSE)

  .p1_raw <- foreach(
    ti        = seq_len(nrow(.p1_grid)),
    .combine  = "list",
    .multicombine = TRUE,
    .packages = .pkgs,
    .export   = c(.local_fns, ".p1_grid", "SimPerBatch", "AlphaMRCD")
  ) %dopar% {
    res <- SimulationT2Chart(
      Observation, NumberVariable, SimPerBatch,
      mu, SigmaCorr, AlphaMRCD, .p1_grid$method[ti]
    )
    list(method = .p1_grid$method[ti],
         T2Total   = res$T2Total,
         T2Max     = res$T2Max,
         LenMatrix = ncol(res$T2Matrix))
  }

  # Combinar lotes por método: T2Total y T2Max son concatenaciones directas.
  valuesT2Total <- matrix(vector("list", 3L * 4L), nrow = 3L, ncol = 4L)
  for (.m_i in seq_along(.methods1)) {
    .m       <- .methods1[.m_i]
    .idx     <- which(sapply(.p1_raw, `[[`, "method") == .m)
    .T2Total <- unlist(lapply(.p1_raw[.idx], `[[`, "T2Total"))
    .T2Max   <- unlist(lapply(.p1_raw[.idx], `[[`, "T2Max"))
    .Len     <- .p1_raw[[.idx[1L]]]$LenMatrix
    valuesT2Total[[1L, .m_i]] <- matrix(.T2Total, ncol = .Len)
    valuesT2Total[[2L, .m_i]] <- .T2Total
    valuesT2Total[[3L, .m_i]] <- .T2Max
  }

  # MRCD
  ValuesMRCDT2Matrix <- valuesT2Total[1, 1]
  ValuesMRCDT2Total <- valuesT2Total[2, 1]
  ValuesMRCDT2Max <- valuesT2Total[3, 1]
  KernelMethodMRCD <- MethodUCLKernel(ValuesMRCDT2Total[[1]], AlphaPFA)
  UCLMRCD <- qemp(p = 1 - AlphaPFA, obs = ValuesMRCDT2Total[[1]])
  UCLMaxMRCD <- qemp(p = 1 - AlphaPFA, obs = ValuesMRCDT2Max[[1]])
  UCLKernelMRCD <- KernelMethodMRCD$UCL
  SignalProbabilityT2MRCD <- SignalProbability(ValuesMRCDT2Matrix[[1]], UCLMRCD, UCLMaxMRCD, UCLKernelMRCD)

  # T2MOD
  ValuesT2MODT2Matrix <- valuesT2Total[1, 2]
  ValuesT2MODT2Total <- valuesT2Total[2, 2]
  ValuesT2MODT2Max <- valuesT2Total[3, 2]
  KernelMethodT2MOD <- MethodUCLKernel(ValuesT2MODT2Total[[1]], AlphaPFA)
  UCLT2MOD <- qemp(p = 1 - AlphaPFA, obs = ValuesT2MODT2Total[[1]])
  UCLMaxT2MOD <- qemp(p = 1 - AlphaPFA, obs = ValuesT2MODT2Max[[1]])
  UCLKernelT2MOD <- KernelMethodT2MOD$UCL
  SignalProbabilityT2MOD <- SignalProbability(ValuesT2MODT2Matrix[[1]], UCLT2MOD, UCLMaxT2MOD, UCLKernelT2MOD)

  # EBADIZI
  ValuesEBADIZIT2Matrix <- valuesT2Total[1, 4]
  ValuesEBADIZIT2Total <- valuesT2Total[2, 4]
  ValuesEBADIZIT2Max <- valuesT2Total[3, 4]
  KernelMethodEBADIZI <- MethodUCLKernel(ValuesEBADIZIT2Total[[1]], AlphaPFA)
  UCLEBADIZI <- qemp(p = 1 - AlphaPFA, obs = ValuesEBADIZIT2Total[[1]])
  UCLMaxEBADIZI <- qemp(p = 1 - AlphaPFA, obs = ValuesEBADIZIT2Max[[1]])
  UCLKernelEBADIZI <- KernelMethodEBADIZI$UCL
  SignalProbabilityT2EBADIZI <- SignalProbability(ValuesEBADIZIT2Matrix[[1]], UCLEBADIZI, UCLMaxEBADIZI, UCLKernelEBADIZI)

  # Parte 2 Calculo del parametro de no centralidad
  ArrayRhoShift <- list(
    mu,
    rep(1 / 100, NumberVariable),
    rep(5 / 100, NumberVariable),
    rep(10 / 100, NumberVariable),
    rep(15 / 100, NumberVariable),
    rep(25 / 100, NumberVariable),
    rep(50 / 100, NumberVariable),
    rep(60 / 100, NumberVariable),
    rep(75 / 100, NumberVariable),
    rep(90 / 100, NumberVariable),
    rep(1, NumberVariable)
  )


  Inverse <- solve(SigmaCorr)
  MatrixDeltaMRCD <- matrix(, ncol = 4)
  MatrixDeltaT2MOD <- matrix(, ncol = 4)
  MatrixDeltaEBADIZI <- matrix(, ncol = 4)
  MatrixDeltaMRCDHesti <- matrix(, ncol = 4)


  # UCL lookup keyed by method — avoids 3 sequential parallel loops
  .ucl_map <- list(
    MRCD    = c(UCLMRCD,    UCLMaxMRCD,    UCLKernelMRCD),
    T2MOD   = c(UCLT2MOD,   UCLMaxT2MOD,   UCLKernelT2MOD),
    EBADIZI = c(UCLEBADIZI, UCLMaxEBADIZI, UCLKernelEBADIZI)
  )
  .delta_methods <- c("MRCD", "T2MOD", "EBADIZI")
  .n_shifts  <- length(ArrayRhoShift)
  .n_methods <- length(.delta_methods)

  .delta_fns <- c("SimulationT2ChartOutliers", "AlgorithmMDPCFPart1", "AlgorithmRoMDP",
                  "rmvnorm_tpu", "mahalanobis_tpu_batch", "SignalProbability")

  # Grid plano: métodos × shifts × lotes = 3 × 11 × 10 = 330 tareas.
  # La probabilidad de señal se promedia entre lotes (lotes iguales → media = total).
  .n_ms      <- .n_methods * .n_shifts
  .n_tasks_d <- .n_ms * NumBatches

  .delta_raw <- foreach(
    task_idx  = seq_len(.n_tasks_d),
    .combine  = "rbind",
    .packages = .pkgs,
    .export   = c(.delta_fns,
                  "ArrayRhoShift", ".ucl_map", ".delta_methods", ".n_shifts", ".n_ms",
                  "Observation", "NumberVariable", "SimPerBatchDelta",
                  "mu", "SigmaCorr", "Percentoutliers", "AlphaMRCD", "Inverse")
  ) %dopar% {
    ms_idx  <- ((task_idx - 1L) %% .n_ms) + 1L
    m_idx   <- ((ms_idx   - 1L) %/% .n_shifts) + 1L
    s_idx   <- ((ms_idx   - 1L) %% .n_shifts) + 1L
    typeM   <- .delta_methods[m_idx]
    shiftmu <- ArrayRhoShift[[s_idx]]
    ucls    <- .ucl_map[[typeM]]
    DeltaNCP <- sqrt(t(shiftmu - mu) %*% Inverse %*% (shiftmu - mu))
    SimulationT2ChartOutliers(
      Observation, NumberVariable, SimPerBatchDelta,
      mu, SigmaCorr, shiftmu, Percentoutliers, AlphaMRCD, DeltaNCP,
      ucls[1], ucls[2], ucls[3], typeM
    )
  }

  # Promediar probabilidades de señal entre lotes; DeltaNCP es idéntico → tomar fila 1.
  .delta_all <- matrix(0, .n_ms, 4L)
  for (.ms_i in seq_len(.n_ms)) {
    .rows <- .delta_raw[seq(.ms_i, .n_tasks_d, by = .n_ms), , drop = FALSE]
    .delta_all[.ms_i, ] <- c(.rows[1L, 1L], colMeans(.rows[, 2:4, drop = FALSE]))
  }

  MatrixDeltaMRCD    <- .delta_all[seq_len(.n_shifts), ]
  MatrixDeltaT2MOD   <- .delta_all[seq_len(.n_shifts) + .n_shifts, ]
  MatrixDeltaEBADIZI <- .delta_all[seq_len(.n_shifts) + 2L * .n_shifts, ]

  save.image(path)

  drive_upload(path, path = "Colab10000simulacion/")

}

stopCluster(cl)
## nolint end


Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): SimulationT2Chart, rmvnorm_tpu, mahalanobis_tpu_batch, AlgorithmMDPCFPart1, AlgorithmRoMDP, SimPerBatch, AlphaMRCD”
Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): SimulationT2ChartOutliers, AlgorithmMDPCFPart1, AlgorithmRoMDP, rmvnorm_tpu, mahalanobis_tpu_batch, SignalProbability, ArrayRhoShift, Observation, NumberVariable, SimPerBatchDelta, mu, SigmaCorr, Percentoutliers, AlphaMRCD, Inverse”
Auto-refreshing stale OAuth token.

Local file:

• /content/SinalprobabilityNormal100x200x0.2x10000.RData

Uploaded into Drive file:

• SinalprobabilityNormal100x200x0.2x10000.RData
  <id: 1vMoChTZ4eM6jhEzOSlIddM8A9T7T7-Px>

With MIME type:

• application/x-gzip

Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): SimulationT2Chart, rmvnorm_tpu, mahalanobis_tpu_batch, AlgorithmMDPCFPart1, Al